# Modern Coding Practice — Rate Limiter (Amazon FAR style)

Same format as the Morse notebook: one domain, the interviewer keeps adding constraints. Parts 1-3
build the core (concurrency arrives at Part 3); Parts 4-5 are harder stretch tasks (error/noise
tolerance, then scale + multiprocessing). Fill the stubs, run each test cell, peek at the collapsed
`ref_` solutions only after you have tried.

**Clocks are injected** (`now` is passed in) so tests are deterministic — call that out as a
testability win in the interview.

---

## Part 1 — Token bucket

`TokenBucket(capacity, refill_per_sec)` with `allow(now, cost=1) -> bool`. A bucket holds up to
`capacity` tokens, refilling continuously at `refill_per_sec`. `allow` refills based on time since
the last call, then consumes `cost` tokens if available.

**Lock down:** float vs int tokens? refill capped at capacity (no infinite accrual). What does the
first-ever call see (start full)? Why a token bucket allows *bursts* up to capacity but a fixed
window does not.

In [ ]:
class TokenBucket:
    def __init__(self, capacity, refill_per_sec):
        raise NotImplementedError

    def allow(self, now, cost=1) -> bool:
        """Refill since last call, then consume `cost` tokens if available."""
        raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    b = TokenBucket(capacity=2, refill_per_sec=1)
    assert b.allow(now=0)          # start full: 2 -> 1
    assert b.allow(now=0)          # 1 -> 0
    assert not b.allow(now=0)      # empty
    assert b.allow(now=1)          # +1 token over 1s
    assert not b.allow(now=1)
    assert b.allow(now=10)         # refill capped at capacity
    assert b.allow(now=10)
    assert not b.allow(now=10)
    print('Part 1 OK')

_t1()

## Part 2 — Sliding-window limiter

`SlidingWindowLimiter(limit, window)` with `allow(now) -> bool`: at most `limit` events in any
trailing `window` seconds. Keep timestamps of admitted events; evict those older than `now - window`.

**Lock down:** sliding *log* (exact, O(events) memory) vs sliding *counter* (approximate, O(1)).
Boundary: is an event exactly `window` old still counted? (We evict `<= now - window`.)

In [ ]:
from collections import deque


class SlidingWindowLimiter:
    def __init__(self, limit, window):
        raise NotImplementedError

    def allow(self, now) -> bool:
        raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    s = SlidingWindowLimiter(limit=3, window=10)
    assert s.allow(0)
    assert s.allow(1)
    assert s.allow(2)
    assert not s.allow(3)      # window holds {0,1,2}, full
    assert s.allow(11)         # evict <=1 -> {2}, admit -> {2,11}
    assert s.allow(11)         # -> {2,11,11}
    assert not s.allow(11)     # full
    assert s.allow(12)         # evict <=2 -> {11,11}, admit
    print('Part 2 OK')

_t2()

## Part 3 — Thread-safe shared limiter

Wrap a limiter so N threads can hit it concurrently without corrupting the token count.
`ThreadSafeLimiter(limiter)` delegates `allow(...)` under a lock.

**The point:** with a bucket of capacity C, refill 0, and 100 threads each calling `allow` once,
*exactly* C must succeed. Without the lock you get a read-modify-write race and over-admit.
**Discuss:** lock granularity, why this is the classic check-then-act bug, and that this is
blocking/coordination work (threads fine) vs CPU-bound (Part 5 uses processes).

In [ ]:
import threading


class ThreadSafeLimiter:
    def __init__(self, limiter):
        raise NotImplementedError

    def allow(self, *args, **kwargs) -> bool:
        raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    lim = ThreadSafeLimiter(TokenBucket(capacity=50, refill_per_sec=0))
    results, rlock, barrier = [], threading.Lock(), threading.Barrier(100)

    def worker():
        barrier.wait()                 # maximize contention
        ok = lim.allow(now=0)
        with rlock:
            results.append(ok)

    ts = [threading.Thread(target=worker) for _ in range(100)]
    for t in ts:
        t.start()
    for t in ts:
        t.join()
    assert sum(results) == 50, sum(results)
    print('Part 3 OK')

_t3()

## Part 4 (stretch) — Robust to clock skew & bad input

Real clocks jump backwards (NTP correction, VM pauses) and callers send junk. Build
`RobustTokenBucket(capacity, refill_per_sec)` whose `allow(now, cost=1)`:

- **never grants free tokens when `now` goes backwards** — clamp elapsed at 0 and never move the
  internal clock backwards.
- **rejects bad cost**: `cost <= 0` raises `ValueError`.
- **never deadlocks on an impossible request**: `cost > capacity` simply returns `False` without
  corrupting state.

**Discuss:** monotonic vs wall clock, why `last = max(last, now)`, and how you would surface
"this request can never succeed" to the caller.

In [ ]:
class RobustTokenBucket:
    def __init__(self, capacity, refill_per_sec):
        raise NotImplementedError

    def allow(self, now, cost=1) -> bool:
        raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    b = RobustTokenBucket(capacity=5, refill_per_sec=1)
    assert b.allow(now=100, cost=5)        # drain
    assert not b.allow(now=100, cost=1)
    assert not b.allow(now=50, cost=1)     # clock went backwards -> no free refill
    assert b.allow(now=105, cost=5)        # +5 over 5s from last-seen 100 -> full again
    assert not b.allow(now=1000, cost=99)  # impossible request: deny, do not corrupt/block
    assert b.allow(now=1000, cost=5)       # state intact: refilled to capacity
    for bad in (0, -1):
        try:
            b.allow(now=1000, cost=bad)
        except ValueError:
            pass
        else:
            raise AssertionError('expected ValueError for cost=%r' % bad)
    print('Part 4 OK')

_t4()

## Part 5 (stretch) — Per-key limiter + parallel replay

**(a)** `KeyedLimiter(capacity, refill_per_sec)` keeps an independent bucket per key, created
lazily, thread-safe: `allow(key, now, cost=1)`. Keys must not interfere.

**(b)** `parallel_replay(logs, capacity, refill_per_sec) -> dict[key, allowed_count]` replays many
recorded request logs (CPU-bound) across processes. `logs` is `dict[key, list[(now, cost)]]`. Use
`ProcessPoolExecutor`; the worker is `ratelimit_workers.replay`.

**Discuss:** sharding to reduce lock contention (lock per key/shard vs one global lock), GIL (replay
is CPU-bound -> processes), pickling cost of shipping logs, and that each key replays independently
(embarrassingly parallel).

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import ratelimit_workers


class KeyedLimiter:
    def __init__(self, capacity, refill_per_sec):
        raise NotImplementedError

    def allow(self, key, now, cost=1) -> bool:
        raise NotImplementedError


def parallel_replay(logs, capacity, refill_per_sec) -> dict:
    """Replay each key's log on a fresh bucket across processes; return {key: allowed_count}."""
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    lim = KeyedLimiter(capacity=2, refill_per_sec=0)
    assert lim.allow('a', now=0)
    assert lim.allow('a', now=0)
    assert not lim.allow('a', now=0)      # key 'a' drained
    assert lim.allow('b', now=0)          # key 'b' independent
    logs = {
        'a': [(0, 1)] * 5,                # capacity 2, no refill -> 2
        'b': [(0, 1)],                    # -> 1
        'c': [(t, 1) for t in range(5)],  # refill 1/s keeps pace -> 5
    }
    counts = parallel_replay(logs, capacity=2, refill_per_sec=1)
    assert counts == {'a': 2, 'b': 1, 'c': 5}, counts
    print('Part 5 OK')

_t5()

## Likely follow-ups
- Distributed limiter (Redis token bucket / Lua), clock sync across nodes.
- Fairness: per-tenant quotas, weighted fair queuing, graceful 429 + Retry-After.
- Approximate counters at scale (sliding-window counter, probabilistic).

---
## Reference solutions (don't peek until you've tried)

In [ ]:
from collections import deque
import threading
from concurrent.futures import ProcessPoolExecutor
import ratelimit_workers


class RefTokenBucket:
    def __init__(self, capacity, refill_per_sec):
        self.capacity, self.rate = capacity, refill_per_sec
        self.tokens, self.last = float(capacity), None

    def allow(self, now, cost=1):
        if self.last is None:
            self.last = now
        elapsed = max(0.0, now - self.last)
        self.last = now
        self.tokens = min(self.capacity, self.tokens + elapsed * self.rate)
        if self.tokens >= cost:
            self.tokens -= cost
            return True
        return False


class RefSliding:
    def __init__(self, limit, window):
        self.limit, self.window, self.events = limit, window, deque()

    def allow(self, now):
        while self.events and self.events[0] <= now - self.window:
            self.events.popleft()
        if len(self.events) < self.limit:
            self.events.append(now)
            return True
        return False


class RefThreadSafe:
    def __init__(self, limiter):
        self._lim, self._lock = limiter, threading.Lock()

    def allow(self, *args, **kwargs):
        with self._lock:
            return self._lim.allow(*args, **kwargs)


class RefRobustBucket:
    def __init__(self, capacity, refill_per_sec):
        self.capacity, self.rate = capacity, refill_per_sec
        self.tokens, self.last = float(capacity), None

    def allow(self, now, cost=1):
        if cost <= 0:
            raise ValueError('cost must be positive')
        if self.last is None:
            self.last = now
        elapsed = max(0.0, now - self.last)
        self.last = max(self.last, now)          # never move clock backwards
        self.tokens = min(self.capacity, self.tokens + elapsed * self.rate)
        if self.tokens >= cost:
            self.tokens -= cost
            return True
        return False


class RefKeyedLimiter:
    def __init__(self, capacity, refill_per_sec):
        self.capacity, self.rate = capacity, refill_per_sec
        self._buckets, self._lock = {}, threading.Lock()

    def allow(self, key, now, cost=1):
        with self._lock:                          # discuss: shard this lock per key
            b = self._buckets.get(key)
            if b is None:
                b = RefRobustBucket(self.capacity, self.rate)
                self._buckets[key] = b
            return b.allow(now, cost)


def ref_parallel_replay(logs, capacity, refill_per_sec):
    items = [(k, ev, capacity, refill_per_sec) for k, ev in logs.items()]
    with ProcessPoolExecutor() as ex:
        return dict(ex.map(ratelimit_workers.replay, items))


# sanity-check the references against the same scenarios as the part tests
_b = RefTokenBucket(2, 1)
assert [_b.allow(0), _b.allow(0), _b.allow(0), _b.allow(1)] == [True, True, False, True]
_s = RefSliding(3, 10)
assert [_s.allow(0), _s.allow(1), _s.allow(2), _s.allow(3), _s.allow(11)] == [True, True, True, False, True]
_rb = RefRobustBucket(5, 1)
assert _rb.allow(100, 5) and not _rb.allow(50, 1) and _rb.allow(105, 5)
assert ref_parallel_replay({'a': [(0, 1)] * 5, 'c': [(t, 1) for t in range(5)]}, 2, 1) == {'a': 2, 'c': 5}
print('reference solutions OK')